# LeetCode 362 — Design Hit Counter bằng Python

Link đề: https://leetcode.com/problems/design-hit-counter/ · Độ khó: **Medium** · Chủ đề: Design, Queue, Array, Binary Search, Data Stream

Skeleton LeetCode:

```python
class HitCounter:

    def __init__(self):
        ...

    def hit(self, timestamp: int) -> None:
        ...

    def getHits(self, timestamp: int) -> int:
        ...


# Your HitCounter object will be instantiated and called as such:
# obj = HitCounter()
# obj.hit(timestamp)
# param_2 = obj.getHits(timestamp)
```

Ghi chú: đây là bài **Premium** nên đề được viết theo hiểu biết về bài; hãy đối chiếu lại với đề gốc nếu bạn có tài khoản Premium. Mọi đáp án trong notebook đã được kiểm chứng bằng code (kể cả bộ chạy chuỗi lệnh mô phỏng cách LeetCode gọi lớp).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu **bài thiết kế lớp** (design problem): LeetCode không gọi một hàm duy nhất mà gọi **một chuỗi lệnh** (`hit`, `getHits`, ...) trên cùng **một đối tượng**, nên đối tượng phải **nhớ trạng thái** giữa các lần gọi (`self.xxx`).
2. Nắm các công cụ Python: `list.append`, `collections.deque` (`append`, `popleft`, `dq[0]`), `bisect.bisect_left`, phép `%` để biến thời gian thành chỉ số ô trong **mảng vòng**.
3. Đi từ cách **lưu mọi lượt hit rồi đếm lại từ đầu** → **hàng đợi `deque` bỏ phần quá hạn** → **chặt nhị phân trên danh sách tăng dần** → **mảng vòng 300 ô** → **hàng đợi nén `(timestamp, số lượt)`**.
4. Hiểu mấu chốt: vì `timestamp` **không giảm**, những lượt hit **cũ hơn 300 giây** chỉ nằm ở **đầu** dãy và **không bao giờ cần lại** nữa; cửa sổ thời gian là khoảng **nửa mở** `(t − 300, t]`.
5. Trả lời được câu **follow-up** (hàng triệu hit mỗi giây): vì sao nén theo giây hoặc dùng mảng 300 ô giúp bộ nhớ không phụ thuộc số lượt hit; tránh các lỗi kinh điển (lệch 1 ở biên cửa sổ, quên dọn hit quá hạn, ô vòng chứa dữ liệu cũ).

## 2. Đề bài & ví dụ

Thiết kế một **bộ đếm lượt truy cập (hit counter)** đếm số lượt hit nhận được trong **5 phút gần nhất** (tức **300 giây** gần nhất).

Hệ thống nhận tham số `timestamp` (đơn vị **giây**). Các lời gọi được thực hiện **theo thứ tự thời gian** (`timestamp` **không giảm**). Có thể có **nhiều hit cùng một `timestamp`**.

Cài đặt lớp `HitCounter`:

- `HitCounter()` — khởi tạo đối tượng.
- `void hit(int timestamp)` — ghi nhận một lượt hit tại `timestamp`.
- `int getHits(int timestamp)` — trả về số lượt hit trong 300 giây gần nhất tính từ `timestamp`.

**Ví dụ 1**

```
Input
["HitCounter", "hit", "hit", "hit", "getHits", "hit", "getHits", "getHits"]
[[], [1], [2], [3], [4], [300], [300], [301]]
Output
[null, null, null, null, 3, null, 4, 3]
```

Giải thích: hit ở giây 1, 2, 3; `getHits(4)` = `3`; hit ở giây 300; `getHits(300)` = `4` (gồm các giây 1, 2, 3, 300); `getHits(301)` = `3` (giây 1 đã "hết hạn", còn 2, 3, 300).

**Ràng buộc:** `1 <= timestamp <= 2 * 10^9`; mọi lời gọi theo thứ tự thời gian (`timestamp` không giảm); có **tối đa `300`** lời gọi `hit` và `getHits`.

**Follow-up:** nếu số lượt hit **mỗi giây rất lớn** thì thiết kế của bạn còn chạy tốt không?

Câu hỏi quan trọng nhất của bài: **"trong 300 giây gần nhất" nghĩa chính xác là những `timestamp` nào?** Từ ví dụ: tại `t = 301` giây `1` bị loại nhưng giây `2` còn; tại `t = 300` giây `1` còn. Vậy cửa sổ là `(t − 300, t]`, tức `t − 299 <= ts <= t`. Ta sẽ kiểm tra lại ở mục 3.5.

In [1]:
import json
import random
from collections import deque
from bisect import bisect_left

commands = ["HitCounter", "hit", "hit", "hit", "getHits", "hit", "getHits", "getHits"]
arguments = [[], [1], [2], [3], [4], [300], [300], [301]]
expected_example = [None, None, None, None, 3, None, 4, 3]

for name, arg in zip(commands, arguments):
    print(f"{name}({', '.join(map(str, arg))})")
print("Kết quả mong đợi (null = None):", json.dumps(expected_example))

HitCounter()
hit(1)
hit(2)
hit(3)
getHits(4)
hit(300)
getHits(300)
getHits(301)
Kết quả mong đợi (null = None): [null, null, null, null, 3, null, 4, 3]


## 3. Kiến thức nền

### 3.1. Bài thiết kế lớp: LeetCode gọi như thế nào?

Với bài thường, LeetCode gọi `Solution().method(...)` đúng **một lần**. Với **bài thiết kế**, nó làm thế này:

1. Tạo một đối tượng: `obj = HitCounter()`.
2. Gọi lần lượt các phương thức theo danh sách `commands`/`arguments`, **trên cùng một `obj`**.
3. Ghi kết quả của mỗi lời gọi (phương thức không trả gì được ghi là `null`, tức `None` trong Python).

Vì cùng một đối tượng sống suốt chuỗi lệnh, ta lưu dữ liệu vào **thuộc tính** `self.xxx` (khởi tạo trong `__init__`). Biến cục bộ trong `hit` sẽ **mất** khi hàm kết thúc; `self.xxx` thì **còn**.

Ví dụ nhỏ: một bộ đếm số lần gọi.

In [2]:
class Counter:
    def __init__(self):
        self.n = 0                 # thuộc tính: sống cùng đối tượng

    def tick(self):
        self.n += 1                # cập nhật trạng thái

    def value(self):
        return self.n


c = Counter()
c.tick(); c.tick(); c.tick()
print(c.value())                   # 3  (đối tượng nhớ 3 lần tick)

d = Counter()                      # đối tượng khác -> trạng thái riêng
print(d.value())                   # 0

3
0


Bây giờ ta tự dựng **bộ chạy chuỗi lệnh** `run_commands` để mô phỏng cách LeetCode gọi lớp. Phần tử đầu của `commands` là tên lớp (ta dùng để tạo đối tượng); các phần tử sau là tên phương thức. `getattr(obj, "hit")` lấy phương thức theo tên; `*a` "rải" danh sách tham số thành các đối số.

In [3]:
def run_commands(cls, commands, arguments):
    """Mô phỏng LeetCode: phần tử đầu là hàm dựng, các phần tử sau là phương thức."""
    obj = None
    outputs = []
    for name, a in zip(commands, arguments):
        if obj is None:
            obj = cls(*a)                       # "HitCounter" -> tạo đối tượng
            outputs.append(None)
        else:
            outputs.append(getattr(obj, name)(*a))   # hit(...) trả None, getHits(...) trả số
    return outputs


# Thử bộ chạy với lớp Counter ở trên (đổi tên lệnh cho khớp)
print(run_commands(Counter, ["Counter", "tick", "tick", "value"], [[], [], [], []]))   # [None, None, None, 2]
print(json.dumps(run_commands(Counter, ["Counter", "tick", "value"], [[], [], []])))   # [null, null, 1]

[None, None, None, 2]
[null, null, 1]


### 3.2. `list`: `append`, `len`, truy cập đầu/cuối

`lst.append(x)` thêm `x` vào **cuối** (nhanh). `len(lst)` là số phần tử. `lst[0]` là phần tử đầu, `lst[-1]` là phần tử **cuối**. Vì các `timestamp` đến theo thứ tự không giảm, danh sách các lượt hit luôn **đã được sắp xếp tăng dần** — một tính chất ta sẽ khai thác.

In [4]:
hits = []
for t in [1, 2, 2, 3, 300]:
    hits.append(t)
print(hits, len(hits))            # [1, 2, 2, 3, 300] 5
print(hits[0], hits[-1])          # 1 300  (đầu, cuối)
print(hits == sorted(hits))       # True: không giảm
print(hits[2:])                   # [2, 3, 300]  (cắt phần đuôi)

[1, 2, 2, 3, 300] 5
1 300
True
[2, 3, 300]


### 3.3. `deque`: hàng đợi hai đầu

`list.pop(0)` (xóa phần tử đầu) phải **dịch toàn bộ** phần còn lại một ô nên tốn `O(n)`. `collections.deque` ("double-ended queue") cho phép thêm/xóa ở **cả hai đầu** trong `O(1)`:

- `dq.append(x)` thêm vào cuối; `dq.popleft()` lấy ra và bỏ phần tử **đầu**.
- `dq[0]` xem phần tử đầu, `dq[-1]` xem phần tử cuối; `len(dq)` đếm; `while dq and ...` để dọn.

In [5]:
dq = deque()
for t in [1, 2, 3, 300]:
    dq.append(t)
print(dq, len(dq))                # deque([1, 2, 3, 300]) 4
print(dq[0], dq[-1])              # 1 300
print(dq.popleft(), list(dq))     # 1 [2, 3, 300]
while dq and dq[0] <= 2:          # dọn phần tử đầu thỏa điều kiện (nhớ kiểm tra dq không rỗng!)
    dq.popleft()
print(list(dq))                   # [3, 300]

deque([1, 2, 3, 300]) 4
1 300
1 [2, 3, 300]
[3, 300]


### 3.4. `bisect_left`: tìm vị trí trong danh sách đã sắp xếp

`bisect_left(a, x)` trả về **chỉ số đầu tiên** `i` sao cho `a[i] >= x` (nếu mọi phần tử đều `< x` thì trả `len(a)`). Chạy trong `O(log n)` nhờ **chặt nhị phân**. Khi đó `len(a) - i` chính là **số phần tử `>= x`**.

In [6]:
a = [1, 2, 3, 300]
print(bisect_left(a, 2))          # 1   (a[1] = 2 là phần tử đầu tiên >= 2)
print(bisect_left(a, 4))          # 3   (a[3] = 300 là phần tử đầu tiên >= 4)
print(bisect_left(a, 1000))       # 4   (không có phần tử nào >= 1000 -> len(a))
print(len(a) - bisect_left(a, 2)) # 3   (số phần tử >= 2: 2, 3, 300)

1
3
4
3


### 3.5. Cửa sổ thời gian: khoảng nửa mở `(t − 300, t]`

"300 giây gần nhất tính từ `t`" có **đúng 300 giây khác nhau**: `t − 299, t − 298, ..., t`. Lượt hit ở giây `t − 300` đã **hết hạn**. Kiểm tra từ ví dụ 1:

- `t = 300`: giây `1` có `300 − 1 = 299 < 300` ⇒ còn.
- `t = 301`: giây `1` có `301 − 1 = 300`, không `< 300` ⇒ hết hạn; giây `2` có `299` ⇒ còn.

Vậy hit tại `ts` được tính khi `t − ts < 300`, hay `ts > t − 300`, hay `ts >= t − 299`.

In [7]:
def in_window(ts, t):
    return t - ts < 300            # tương đương ts > t - 300

print([ts for ts in [1, 2, 3, 300] if in_window(ts, 300)])   # [1, 2, 3, 300] -> 4 hit
print([ts for ts in [1, 2, 3, 300] if in_window(ts, 301)])   # [2, 3, 300]    -> 3 hit
print(in_window(100, 399), in_window(100, 400))               # True False  (100 hết hạn đúng lúc t = 400)

[1, 2, 3, 300]
[2, 3, 300]
True False


### 3.6. Phép `%` để quay vòng chỉ số

`t % 300` luôn cho số trong `0..299`. Các giây cách nhau đúng `300` có **cùng** `t % 300`, nên `300` giây liên tiếp rơi vào **300 ô khác nhau** — đây là cơ sở của **mảng vòng** (ring buffer) ở Cách 4.

In [8]:
for t in [1, 2, 299, 300, 301, 600, 2_000_000_000]:
    print(f"t={t}: ô {t % 300}")
print(sorted({t % 300 for t in range(1000, 1300)}) == list(range(300)))   # 300 giây liên tiếp phủ đủ 300 ô: True

t=1: ô 1
t=2: ô 2
t=299: ô 299
t=300: ô 0
t=301: ô 1
t=600: ô 0
t=2000000000: ô 200
True


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Chạy ví dụ 1 bằng tay. Ta giữ "sổ ghi" gồm các giây có hit và, tại mỗi `getHits(t)`, **gạch** những giây `<= t − 300`.

| Lệnh | Sổ ghi sau lệnh | Cửa sổ `ts > t − 300` | Kết quả |
|---|---|---|---|
| `hit(1)` | `[1]` | | null |
| `hit(2)` | `[1, 2]` | | null |
| `hit(3)` | `[1, 2, 3]` | | null |
| `getHits(4)` | `[1, 2, 3]` | `ts > −296` ⇒ cả ba | `3` |
| `hit(300)` | `[1, 2, 3, 300]` | | null |
| `getHits(300)` | `[1, 2, 3, 300]` | `ts > 0` ⇒ cả bốn | `4` |
| `getHits(301)` | `[1, 2, 3, 300]` | `ts > 1` ⇒ `2, 3, 300` | `3` |

Quan sát: ta **không cần sổ ghi giữ mãi** những giây đã hết hạn; hơn nữa chúng luôn là **những phần tử đầu** của sổ (vì giây nhỏ nhất ở đầu).

In [9]:
log = [1, 2, 3, 300]
for t in [300, 301]:
    alive = [ts for ts in log if ts > t - 300]
    print(f"getHits({t}): còn sống {alive} -> {len(alive)}")

getHits(300): còn sống [1, 2, 3, 300] -> 4
getHits(301): còn sống [2, 3, 300] -> 3


## 5. Cách 1 — Brute Force / baseline: lưu mọi hit, đếm lại toàn bộ

**Ý tưởng.** `hit` chỉ `append(timestamp)` vào danh sách. `getHits(t)` duyệt **toàn bộ** danh sách và đếm những `ts` thỏa `t − ts < 300`. Đúng đề nhất, không tận dụng việc timestamp tăng dần.

In [10]:
class HitCounterBrute:
    def __init__(self):
        self.ts = []                                   # mọi lượt hit từ trước đến nay

    def hit(self, timestamp: int) -> None:
        self.ts.append(timestamp)

    def getHits(self, timestamp: int) -> int:
        count = 0
        for x in self.ts:                              # duyệt TẤT CẢ
            if timestamp - x < 300:
                count += 1
        return count


print(run_commands(HitCounterBrute, commands, arguments))     # [None, None, None, None, 3, None, 4, 3]
assert run_commands(HitCounterBrute, commands, arguments) == expected_example

[None, None, None, None, 3, None, 4, 3]


**Độ phức tạp (Cách 1).** Gọi `n` là số lượt hit đã ghi. `hit`: **`O(1)`**; `getHits`: **`O(n)`** vì quét toàn bộ. **Space `O(n)`** (không bao giờ xóa).

**Nút thắt:** (1) ta quét cả những lượt hit đã hết hạn từ rất lâu; (2) bộ nhớ chỉ tăng. Nếu hệ thống chạy cả ngày với hàng triệu hit thì rất lãng phí, trong khi câu hỏi chỉ liên quan đến **300 giây cuối**.

## 6. Ý tưởng thuật toán / bước đột phá

Hai nhận xét:

1. **Đơn điệu:** `timestamp` của mọi lời gọi **không giảm**. Do đó danh sách hit **đã sắp xếp**, và những hit hết hạn tại thời điểm `t` (tức `ts <= t − 300`) tạo thành **một đoạn tiền tố** (prefix) ở **đầu** danh sách.
2. **Hết hạn là vĩnh viễn:** nếu `ts` đã hết hạn tại `t` thì còn hết hạn tại mọi `t' >= t` (cửa sổ chỉ trượt về phía phải). Vậy ta có thể **xóa hẳn** chúng.

Hai hệ quả, hai hướng giải:

- **Hàng đợi (deque):** bỏ phần tiền tố hết hạn bằng `popleft`; sau đó `len(dq)` chính là đáp án. Mỗi hit bị xóa **một lần** ⇒ chi phí **khấu hao `O(1)`**.
- **Chặt nhị phân:** không xóa gì, chỉ tìm bằng `bisect_left` vị trí đầu tiên còn sống `ts >= t − 299`; đáp án là phần đuôi.

Hướng thứ ba (cho follow-up): chỉ có **300 giây** trong cửa sổ, nên chỉ cần **300 ô** đếm theo giây (mảng vòng) hoặc **nén** các hit cùng giây thành `(giây, số lượt)`. Bộ nhớ khi đó phụ thuộc **300**, không phụ thuộc số hit.

In [11]:
# Ví dụ nhanh: danh sách đã sắp xếp, tìm ranh giới hết hạn tại t = 301
log = [1, 2, 3, 300]
t = 301
boundary = bisect_left(log, t - 300 + 1)          # chỉ số đầu tiên có ts >= t - 299 = 2
print("ranh giới:", boundary, "-> tiền tố hết hạn:", log[:boundary], "| còn sống:", log[boundary:])
print("đáp án:", len(log) - boundary)             # 3

ranh giới: 1 -> tiền tố hết hạn: [1] | còn sống: [2, 3, 300]
đáp án: 3


## 7. Các cách tiếp theo

### Cách 2 — `deque`: bỏ phần quá hạn ở đầu hàng đợi

**Giải thích từng đoạn.**

1. `self.q = deque()`: hàng đợi các `timestamp` còn có thể còn hiệu lực.
2. `hit(t)`: `self.q.append(t)` (thêm vào cuối, vì `t` không giảm nên thứ tự vẫn tăng).
3. `getHits(t)`: `while self.q and self.q[0] <= t - 300: self.q.popleft()` — xóa mọi hit hết hạn ở đầu. Điều kiện `self.q and ...` tránh lỗi khi hàng đợi rỗng.
4. Trả `len(self.q)`: những gì còn lại đều thỏa `ts > t − 300` (và `ts <= t` vì mọi hit đã xảy ra trước hoặc đúng lúc `t`).

In [12]:
class HitCounterDeque:
    def __init__(self):
        self.q = deque()

    def hit(self, timestamp: int) -> None:
        self.q.append(timestamp)

    def getHits(self, timestamp: int) -> int:
        while self.q and self.q[0] <= timestamp - 300:     # đầu hàng đợi đã hết hạn -> bỏ
            self.q.popleft()
        return len(self.q)


print(run_commands(HitCounterDeque, commands, arguments))
assert run_commands(HitCounterDeque, commands, arguments) == expected_example

[None, None, None, None, 3, None, 4, 3]


In [13]:
# Code nộp LeetCode cho Cách 2 (đổi tên lớp thành HitCounter khi nộp)
from collections import deque


class HitCounter:

    def __init__(self):
        self.q = deque()

    def hit(self, timestamp: int) -> None:
        self.q.append(timestamp)

    def getHits(self, timestamp: int) -> int:
        while self.q and self.q[0] <= timestamp - 300:
            self.q.popleft()
        return len(self.q)


print(run_commands(HitCounter, commands, arguments))

[None, None, None, None, 3, None, 4, 3]


**Độ phức tạp Cách 2.** `hit`: `O(1)`. `getHits`: **khấu hao `O(1)`** — mỗi phần tử được `popleft` **tối đa một lần** trong cả đời đối tượng, nên tổng công `popleft` qua `m` lời gọi là `O(m)`. **Space `O(h)`**, với `h` là số hit nằm trong cửa sổ 300 giây (sau khi dọn).

### Cách 3 — Danh sách tăng dần + chặt nhị phân (`bisect_left`)

**Giải thích.** Không xóa gì. `hit` chỉ `append`. `getHits(t)`: danh sách `ts` đã tăng dần nên tìm bằng `bisect_left(self.ts, t - 299)` chỉ số đầu tiên còn sống (`ts >= t − 299`, tương đương `ts > t − 300` vì là số nguyên); đáp án `len(self.ts) - chỉ_số`. Đây là lời giải chính thức trong bản đề của doocs.

In [14]:
class HitCounterBisect:
    def __init__(self):
        self.ts = []

    def hit(self, timestamp: int) -> None:
        self.ts.append(timestamp)

    def getHits(self, timestamp: int) -> int:
        start = bisect_left(self.ts, timestamp - 300 + 1)   # chỉ số đầu tiên có ts >= t - 299
        return len(self.ts) - start


print(run_commands(HitCounterBisect, commands, arguments))
assert run_commands(HitCounterBisect, commands, arguments) == expected_example

[None, None, None, None, 3, None, 4, 3]


**Độ phức tạp Cách 3.** `hit`: `O(1)`; `getHits`: **`O(log n)`** (chặt nhị phân trên `n` hit đã lưu). **Space `O(n)`** (không xóa). Ưu điểm: rất ngắn, và cho phép hỏi cả những mốc thời gian trong quá khứ (nếu đề cho phép). Nhược điểm: bộ nhớ lớn dần.

### Cách 4 — Mảng vòng 300 ô (bucket theo giây)

**Giải thích từng đoạn.** Chỉ cần nhớ **300 giây gần nhất**, mỗi giây một ô:

1. `times[i]` lưu "giây nào đang chiếm ô `i`", `hits[i]` lưu số lượt hit của giây đó. Ô của giây `t` là `i = t % 300`.
2. `hit(t)`: nếu `times[i] != t` thì ô đang chứa dữ liệu của một giây **cũ** (cách đúng `300`, `600`, ...) ⇒ **ghi đè**: `times[i] = t; hits[i] = 1`. Ngược lại cộng thêm: `hits[i] += 1`.
3. `getHits(t)`: cộng `hits[i]` của những ô còn hiệu lực `t − times[i] < 300`. Ô chưa dùng có `times = 0`, `hits = 0` nên không ảnh hưởng.

Mỗi `getHits` quét đúng `300` ô, **không phụ thuộc số hit**.

In [15]:
class HitCounterRing:
    def __init__(self):
        self.times = [0] * 300          # times[i]: giây đang chiếm ô i
        self.hits = [0] * 300           # hits[i]: số lượt hit trong giây đó

    def hit(self, timestamp: int) -> None:
        i = timestamp % 300
        if self.times[i] != timestamp:  # ô đang chứa giây cũ -> đặt lại
            self.times[i] = timestamp
            self.hits[i] = 1
        else:
            self.hits[i] += 1

    def getHits(self, timestamp: int) -> int:
        total = 0
        for i in range(300):
            if timestamp - self.times[i] < 300:     # giây trong ô còn hiệu lực
                total += self.hits[i]
        return total


print(run_commands(HitCounterRing, commands, arguments))
assert run_commands(HitCounterRing, commands, arguments) == expected_example

[None, None, None, None, 3, None, 4, 3]


**Độ phức tạp Cách 4.** `hit`: `O(1)`; `getHits`: `O(300) = O(1)` (hằng số, không phụ thuộc số hit). **Space `O(300) = O(1)`** — **không đổi** dù có hàng tỉ hit. Đây là câu trả lời tự nhiên cho follow-up.

### Cách 5 — `deque` nén theo giây: `(timestamp, số lượt)` + tổng chạy

**Giải thích.** Nếu nhiều hit cùng một giây, thay vì `append` từng cái ta **gộp**: giữ `[giây, số_lượt]` trong hàng đợi, và biến `total` = tổng số lượt trong hàng đợi.

1. `hit(t)`: nếu đuôi hàng đợi cùng giây `t` thì tăng số lượt; không thì thêm `[t, 1]`. Luôn `total += 1`.
2. `getHits(t)`: bỏ các nhóm ở đầu có giây `<= t − 300`, mỗi nhóm bỏ thì `total -= số_lượt`. Trả `total`.

`getHits` giờ **không cần `len`** mà trả `total`, vì một phần tử của hàng đợi có thể đại diện cho hàng triệu hit.

In [16]:
class HitCounterCompressed:
    def __init__(self):
        self.q = deque()                # mỗi phần tử: [giây, số lượt hit trong giây đó]
        self.total = 0                  # tổng số lượt trong hàng đợi

    def hit(self, timestamp: int) -> None:
        if self.q and self.q[-1][0] == timestamp:
            self.q[-1][1] += 1          # cùng giây với nhóm cuối -> gộp
        else:
            self.q.append([timestamp, 1])
        self.total += 1

    def getHits(self, timestamp: int) -> int:
        while self.q and self.q[0][0] <= timestamp - 300:
            self.total -= self.q.popleft()[1]       # bỏ cả nhóm hết hạn
        return self.total


print(run_commands(HitCounterCompressed, commands, arguments))
assert run_commands(HitCounterCompressed, commands, arguments) == expected_example

[None, None, None, None, 3, None, 4, 3]


**Độ phức tạp Cách 5.** `hit`: `O(1)`; `getHits`: khấu hao `O(1)` (mỗi nhóm bị bỏ một lần). **Space `O(min(h, 300))`** — tối đa **300 nhóm** vì có nhiều nhất 300 giây khác nhau trong cửa sổ, bất kể số lượt hit.

Minh họa follow-up: gửi `200 000` hit trong cùng `3` giây và so kích thước bộ nhớ lưu trữ.

In [17]:
big_stress = [1] * 70000 + [2] * 70000 + [3] * 60000
brute, comp, ring = HitCounterBrute(), HitCounterCompressed(), HitCounterRing()
for t in big_stress:
    brute.hit(t); comp.hit(t); ring.hit(t)

print("số phần tử lưu - brute:", len(brute.ts), "| nén:", len(comp.q), "| vòng:", len(ring.times))
print(brute.getHits(4), comp.getHits(4), ring.getHits(4))       # 200000 200000 200000
print(brute.getHits(302), comp.getHits(302), ring.getHits(302)) # giây 1 hết hạn tại 301, giây 2 hết hạn tại 302 -> chỉ còn giây 3: 60000

số phần tử lưu - brute: 200000 | nén: 3 | vòng: 300


200000 200000 200000


60000 60000 60000


## 8. TRACE từng bước

Ta trace **Cách 2 (`deque`)** trên một chuỗi có biên thú vị: `hit(1), hit(2), hit(2), hit(300), getHits(300), getHits(301), getHits(302), hit(400), getHits(600)`. Quan sát khi nào các phần tử bị `popleft`.

In [18]:
def trace_deque(commands, arguments):
    q = deque()
    for name, a in zip(commands, arguments):
        if name == "hit":
            t = a[0]
            q.append(t)
            print(f"hit({t}):      q = {list(q)}")
        elif name == "getHits":
            t = a[0]
            print(f"getHits({t}): trước dọn q = {list(q)}, ngưỡng hết hạn: ts <= {t - 300}")
            while q and q[0] <= t - 300:
                gone = q.popleft()
                print(f"    bỏ {gone} (vì {gone} <= {t - 300})")
            print(f"    sau dọn q = {list(q)} -> trả {len(q)}")


trace_deque(
    ["hit", "hit", "hit", "hit", "getHits", "getHits", "getHits", "hit", "getHits"],
    [[1], [2], [2], [300], [300], [301], [302], [400], [600]],
)

hit(1):      q = [1]
hit(2):      q = [1, 2]
hit(2):      q = [1, 2, 2]
hit(300):      q = [1, 2, 2, 300]
getHits(300): trước dọn q = [1, 2, 2, 300], ngưỡng hết hạn: ts <= 0
    sau dọn q = [1, 2, 2, 300] -> trả 4
getHits(301): trước dọn q = [1, 2, 2, 300], ngưỡng hết hạn: ts <= 1
    bỏ 1 (vì 1 <= 1)
    sau dọn q = [2, 2, 300] -> trả 3
getHits(302): trước dọn q = [2, 2, 300], ngưỡng hết hạn: ts <= 2
    bỏ 2 (vì 2 <= 2)
    bỏ 2 (vì 2 <= 2)
    sau dọn q = [300] -> trả 1
hit(400):      q = [300, 400]
getHits(600): trước dọn q = [300, 400], ngưỡng hết hạn: ts <= 300
    bỏ 300 (vì 300 <= 300)
    sau dọn q = [400] -> trả 1


Bảng trace tương ứng:

| Lệnh | `q` trước dọn | ngưỡng `t − 300` | Bỏ | `q` sau dọn | Trả |
|---|---|---|---|---|---|
| `hit(1)` | `[1]` | | | `[1]` | null |
| `hit(2)` | `[1, 2]` | | | `[1, 2]` | null |
| `hit(2)` | `[1, 2, 2]` | | | `[1, 2, 2]` | null |
| `hit(300)` | `[1, 2, 2, 300]` | | | | null |
| `getHits(300)` | `[1, 2, 2, 300]` | `0` | không | `[1, 2, 2, 300]` | `4` |
| `getHits(301)` | `[1, 2, 2, 300]` | `1` | `1` | `[2, 2, 300]` | `3` |
| `getHits(302)` | `[2, 2, 300]` | `2` | `2, 2` | `[300]` | `1` |
| `hit(400)` | `[300, 400]` | | | | null |
| `getHits(600)` | `[300, 400]` | `300` | `300` | `[400]` | `1` |

Chú ý `getHits(302)`: hai hit cùng giây `2` hết hạn **cùng lúc**, vòng `while` bỏ cả hai. Tại `getHits(600)`: giây `300` cách `600` đúng `300` nên hết hạn (cửa sổ nửa mở).

Giờ trace **Cách 4 (mảng vòng)** để thấy việc **ghi đè ô** khi giây mới trùng chỉ số: `hit(5)`, `hit(305)` cùng ô `5`.

In [19]:
def trace_ring(commands, arguments):
    times, hits = {}, {}                      # chỉ in các ô đã dùng (mảng thật có 300 ô)
    for name, a in zip(commands, arguments):
        t = a[0]
        i = t % 300
        if name == "hit":
            if times.get(i, 0) != t:
                print(f"hit({t}): ô {i} đang chứa giây {times.get(i, 0)} (cũ) -> ghi đè: giây {t}, 1 lượt")
                times[i], hits[i] = t, 1
            else:
                hits[i] += 1
                print(f"hit({t}): ô {i} cùng giây {t} -> số lượt = {hits[i]}")
        else:
            alive = {j: hits[j] for j in times if t - times[j] < 300}
            print(f"getHits({t}): ô còn hiệu lực {alive} -> trả {sum(alive.values())}")


trace_ring(
    ["hit", "hit", "hit", "getHits", "getHits", "getHits"],
    [[5], [5], [305], [305], [320], [604]],
)

hit(5): ô 5 đang chứa giây 0 (cũ) -> ghi đè: giây 5, 1 lượt
hit(5): ô 5 cùng giây 5 -> số lượt = 2
hit(305): ô 5 đang chứa giây 5 (cũ) -> ghi đè: giây 305, 1 lượt
getHits(305): ô còn hiệu lực {5: 1} -> trả 1
getHits(320): ô còn hiệu lực {5: 1} -> trả 1
getHits(604): ô còn hiệu lực {5: 1} -> trả 1


Nhận xét: tại `hit(305)`, ô `5` đang giữ giây `5` (cũ, cách đúng `300`) nên bị ghi đè — **nếu không kiểm tra `times[i] != t` mà chỉ `hits[i] += 1`** ta sẽ cộng nhầm vào dữ liệu cũ (xem Lỗi 3 ở mục 13). Tại `getHits(604)`: giây `305` có `604 − 305 = 299 < 300` nên còn; không có ô nào khác còn hiệu lực.

## 9. Code nộp LeetCode

Bản nộp dùng **Cách 2 (`deque`)**: ngắn, khấu hao `O(1)`. (Với follow-up hãy dùng Cách 5 hoặc Cách 4.) LeetCode đã import sẵn một số thư viện; ta vẫn `from collections import deque` cho chắc.

In [20]:
from collections import deque


class HitCounter:

    def __init__(self):
        self.q = deque()                      # các timestamp còn có thể còn hiệu lực (tăng dần)

    def hit(self, timestamp: int) -> None:
        self.q.append(timestamp)

    def getHits(self, timestamp: int) -> int:
        while self.q and self.q[0] <= timestamp - 300:    # hết hạn: cách >= 300 giây
            self.q.popleft()
        return len(self.q)


# Your HitCounter object will be instantiated and called as such:
# obj = HitCounter()
# obj.hit(timestamp)
# param_2 = obj.getHits(timestamp)

print(json.dumps(run_commands(HitCounter, commands, arguments)))
assert run_commands(HitCounter, commands, arguments) == expected_example

[null, null, null, null, 3, null, 4, 3]


## 10. Vì sao thuật toán đúng

Gọi `q` là hàng đợi sau khi xử lý các lời gọi đến thời điểm `t`.

- **Bất biến 1 (sắp xếp):** `q` luôn tăng dần (không giảm), vì ta chỉ `append` các `timestamp` không giảm.
- **Bất biến 2 (không mất hit còn sống):** ta chỉ `popleft` phần tử `x` khi `x <= t − 300`. Vì mọi lời gọi sau có `t' >= t`, ta cũng có `x <= t' − 300`, tức `x` cũng hết hạn ở các lần sau. Vậy **không bao giờ** xóa nhầm hit còn dùng được.
- **Bất biến 3 (không giữ hit chết khi trả lời):** vòng `while` dừng khi `q[0] > t − 300` hoặc `q` rỗng. Vì `q` tăng dần, **mọi** phần tử còn lại đều `> t − 300`, tức hết hạn không còn. Và mọi phần tử đều `<= t` (hit xảy ra trước hoặc cùng lúc, vì lời gọi theo thứ tự thời gian).
- **Kết luận:** những phần tử còn lại **đúng bằng** tập hit với `t − 300 < ts <= t`, nên `len(q)` là đáp án.
- **Khấu hao:** mỗi `append` thêm đúng một phần tử và mỗi phần tử bị `popleft` tối đa một lần ⇒ tổng công qua `m` lời gọi là `O(m)`.

Với Cách 4: một giây `s` chỉ có thể ở ô `s % 300`; trong cửa sổ `(t − 300, t]` có đúng 300 giây liên tiếp nên **không có hai giây nào cùng ô**, do đó mỗi ô chứa tối đa một giây "sống". Ô mang giây đã hết hạn bị loại ở `getHits` bằng điều kiện `t − times[i] < 300`.

## 11. Phân tích Time / Space Complexity

Gọi `n` là số hit đã ghi, `h` là số hit trong cửa sổ 300 giây, `m` là số lời gọi.

| Cách | `hit` | `getHits` | Space |
|---|---|---|---|
| 1. Lưu hết, đếm lại | `O(1)` | `O(n)` | `O(n)` |
| 2. `deque` bỏ quá hạn | `O(1)` | khấu hao `O(1)` | `O(h)` |
| 3. `bisect` trên danh sách | `O(1)` | `O(log n)` | `O(n)` |
| 4. Mảng vòng 300 ô | `O(1)` | `O(300) = O(1)` | `O(300) = O(1)` |
| 5. `deque` nén `(giây, số lượt)` | `O(1)` | khấu hao `O(1)` | `O(min(h, 300))` |

Với ràng buộc của đề (`<= 300` lời gọi) mọi cách đều tức thì; sự khác nhau chỉ lộ ra khi **hệ thống chạy thật lâu** hoặc **hit mỗi giây rất nhiều** (follow-up).

## 12. Bộ test

Ta gom các cách vào `impls` và so sánh kết quả của **chuỗi lệnh** với kỳ vọng. Với bài thiết kế, ta tạo một hàm `check_all(commands, arguments, expected, label)`.

In [21]:
impls = {
    "brute": HitCounterBrute,
    "deque": HitCounterDeque,
    "bisect": HitCounterBisect,
    "ring": HitCounterRing,
    "compressed": HitCounterCompressed,
    "leetcode": HitCounter,
}


def check_all(commands, arguments, expected, label):
    for key, cls in impls.items():
        actual = run_commands(cls, commands, arguments)
        assert actual == expected, (label, key, actual, expected)
    print("PASS", label)


check_all(commands, arguments, expected_example, "ví dụ 1 của đề")

PASS ví dụ 1 của đề


### 12.1. Các trường hợp biên

Chú ý các mốc: hit tại `t` còn tính ở `t + 299` nhưng **hết hạn ở `t + 300`**; nhiều hit cùng giây; `getHits` khi chưa có hit; `timestamp` cực lớn.

In [22]:
def seq(*ops):
    """seq(('hit', 1), ('getHits', 5)) -> (commands, arguments) kèm lệnh tạo đối tượng."""
    return ["HitCounter"] + [o[0] for o in ops], [[]] + [[o[1]] for o in ops]


# chưa có hit nào
c, a = seq(("getHits", 1), ("getHits", 500))
check_all(c, a, [None, 0, 0], "chưa có hit")

# biên cửa sổ: hit tại 1 còn tính ở 300, hết hạn ở 301
c, a = seq(("hit", 1), ("getHits", 299), ("getHits", 300), ("getHits", 301), ("getHits", 302))
check_all(c, a, [None, None, 1, 1, 0, 0], "biên 299/300/301")

# nhiều hit cùng giây
c, a = seq(("hit", 5), ("hit", 5), ("hit", 5), ("getHits", 5), ("getHits", 304), ("getHits", 305))
check_all(c, a, [None, None, None, None, 3, 3, 0], "3 hit cùng giây")

# getHits gọi nhiều lần cùng thời điểm không làm đổi kết quả
c, a = seq(("hit", 10), ("getHits", 10), ("getHits", 10), ("getHits", 10))
check_all(c, a, [None, None, 1, 1, 1], "getHits lặp lại")

# hit rồi getHits cùng timestamp
c, a = seq(("hit", 100), ("getHits", 100))
check_all(c, a, [None, None, 1], "hit rồi getHits cùng giây")

# timestamp lớn nhất theo đề
base = 2 * 10**9 - 400
c, a = seq(("hit", base), ("hit", base + 100), ("getHits", base + 299), ("getHits", base + 300), ("getHits", base + 400))
check_all(c, a, [None, None, None, 2, 1, 0], "timestamp sát 2*10^9")

PASS chưa có hit
PASS biên 299/300/301
PASS 3 hit cùng giây
PASS getHits lặp lại
PASS hit rồi getHits cùng giây
PASS timestamp sát 2*10^9


### 12.2. Phản ví dụ cho các cách làm sai

Hai lỗi phổ biến: (a) **lệch 1** — dùng `ts >= t − 300` (tính cả `ts = t − 300`); (b) **quên bỏ hit quá hạn** nhưng cũng không giới hạn đếm (trả tổng số hit).

In [23]:
class WrongOffByOne:
    def __init__(self):
        self.q = deque()

    def hit(self, timestamp):
        self.q.append(timestamp)

    def getHits(self, timestamp):
        while self.q and self.q[0] < timestamp - 300:        # SAI: nên là <=
            self.q.popleft()
        return len(self.q)


class WrongNoExpire:
    def __init__(self):
        self.n = 0

    def hit(self, timestamp):
        self.n += 1

    def getHits(self, timestamp):
        return self.n                                         # SAI: không bỏ hit cũ


c, a = seq(("hit", 1), ("getHits", 301))
print("đúng:", run_commands(HitCounter, c, a)[-1], "| lệch 1:", run_commands(WrongOffByOne, c, a)[-1],
      "| không bỏ cũ:", run_commands(WrongNoExpire, c, a)[-1])
assert run_commands(WrongOffByOne, c, a) != run_commands(HitCounter, c, a)
assert run_commands(WrongNoExpire, c, a) != run_commands(HitCounter, c, a)

đúng: 0 | lệch 1: 1 | không bỏ cũ: 1


### 12.3. Đối chiếu ngẫu nhiên (có seed) — oracle độc lập

Oracle: **một danh sách lưu toàn bộ hit**, mỗi `getHits` đếm bằng `sum(1 for x in ts if x > t - 300)` — không dùng cấu trúc thông minh nào. Ta sinh chuỗi lệnh **đúng ràng buộc**: tối đa `300` lời gọi, `timestamp` không giảm, trong `[1, 2·10^9]`, với bước nhảy hay rơi vào các mốc nguy hiểm `0, 1, 299, 300, 301`.

In [24]:
def oracle(ops):
    ts, out = [], [None]
    for name, t in ops:
        if name == "hit":
            ts.append(t)
            out.append(None)
        else:
            out.append(sum(1 for x in ts if x > t - 300))
    return out


def random_ops(rng, n_calls, start_hi, steps):
    t = rng.randint(1, start_hi)
    ops = []
    for _ in range(n_calls):
        t += rng.choice(steps)
        assert 1 <= t <= 2 * 10**9
        ops.append((rng.choice(["hit", "hit", "getHits"]), t))
    return ops


rng = random.Random(362)
step_sets = [
    [0, 0, 1],                          # dày đặc, nhiều hit cùng giây
    [0, 1, 5, 50, 299, 300, 301],       # bước quanh biên cửa sổ
    [0, 100, 299, 300, 301, 1000],      # bước lớn
]
for trial in range(1500):
    n_calls = rng.randint(1, 300)
    ops = random_ops(rng, n_calls, rng.choice([1, 10, 10**9]), rng.choice(step_sets))
    c, a = seq(*ops)
    exp = oracle(ops)
    for key, cls in impls.items():
        assert run_commands(cls, c, a) == exp, (key, ops)
print("PASS 1500 chuỗi lệnh ngẫu nhiên đúng ràng buộc (<= 300 lời gọi)")

PASS 1500 chuỗi lệnh ngẫu nhiên đúng ràng buộc (<= 300 lời gọi)


### 12.4. Kiểm tra quy mô lớn (follow-up)

Ngoài ràng buộc của đề, ta thử `100 000` lời gọi để chắc rằng các cách "khấu hao" vẫn đúng và nhanh. Brute Force bị bỏ ra vì `O(n)` mỗi lần `getHits`.

In [25]:
import time

rng = random.Random(3620)
ops = random_ops(rng, 100000, 10**6, [0, 0, 0, 1, 2, 5])
c, a = seq(*ops)
results = {}
for key in ["deque", "bisect", "ring", "compressed", "leetcode"]:
    start = time.perf_counter()
    results[key] = run_commands(impls[key], c, a)
    print(f"{key:11s} xong trong {time.perf_counter() - start:.2f}s")
ref = results["deque"]
assert all(results[k] == ref for k in results)
print("PASS: 5 cách cho cùng kết quả trên 100000 lời gọi; max getHits =", max(x for x in ref if x is not None))

deque       xong trong 0.02s
bisect      xong trong 0.02s


ring        xong trong 0.40s
compressed  xong trong 0.02s
leetcode    xong trong 0.02s
PASS: 5 cách cho cùng kết quả trên 100000 lời gọi; max getHits = 225


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Lệch 1 ở biên cửa sổ (`>= t − 300` thay vì `> t − 300`)

In [26]:
class SaiLech1:
    def __init__(self):
        self.q = deque()

    def hit(self, timestamp):
        self.q.append(timestamp)

    def getHits(self, timestamp):
        while self.q and self.q[0] < timestamp - 300:     # bỏ khi ts < t-300, giữ lại ts == t-300
            self.q.popleft()
        return len(self.q)


c, a = seq(("hit", 1), ("hit", 2), ("hit", 3), ("getHits", 4), ("hit", 300), ("getHits", 300), ("getHits", 301))
print("sai:   ", run_commands(SaiLech1, c, a))
print("đúng:  ", run_commands(HitCounter, c, a))

sai:    [None, None, None, None, 3, None, 4, 4]
đúng:   [None, None, None, None, 3, None, 4, 3]


**Vì sao sai:** hit tại `ts = t − 300` đã cách đúng `300` giây, tức **không còn** trong "300 giây gần nhất" theo ví dụ (`getHits(301)` phải bỏ giây `1`). Code sai trả `4` thay vì `3` ở lệnh cuối. **Cách sửa:** bỏ khi `ts <= t − 300`. Mẹo: luôn thử với ví dụ có hit đúng ở biên `t − 300` và `t − 299`.

### Lỗi 2: Chỉ dọn ở `hit`, không dọn ở `getHits`

In [27]:
class SaiDonOHit:
    def __init__(self):
        self.q = deque()

    def hit(self, timestamp):
        while self.q and self.q[0] <= timestamp - 300:    # dọn khi hit...
            self.q.popleft()
        self.q.append(timestamp)

    def getHits(self, timestamp):
        return len(self.q)                                 # ...nhưng getHits không dọn


c, a = seq(("hit", 1), ("getHits", 400))
print("sai:", run_commands(SaiDonOHit, c, a), "| đúng:", run_commands(HitCounter, c, a))

sai: [None, None, 1] | đúng: [None, None, 0]


**Vì sao sai:** `getHits(400)` được gọi mà **chưa có hit nào mới** để kích hoạt việc dọn; hit ở giây `1` đáng ra đã hết hạn nhưng vẫn nằm trong hàng đợi, nên trả `1` thay vì `0`. **Cách sửa:** dọn **ngay trong `getHits`** theo `timestamp` của chính lời gọi đó (có thể dọn cả ở `hit` nếu muốn tiết kiệm bộ nhớ, nhưng `getHits` bắt buộc phải dọn).

### Lỗi 3: Mảng vòng không kiểm tra ô chứa giây cũ

In [28]:
class SaiRing:
    def __init__(self):
        self.times = [0] * 300
        self.hits = [0] * 300

    def hit(self, timestamp):
        i = timestamp % 300
        self.times[i] = timestamp          # cập nhật giây...
        self.hits[i] += 1                  # ...nhưng cộng dồn lên số lượt CŨ (không reset)

    def getHits(self, timestamp):
        return sum(self.hits[i] for i in range(300) if timestamp - self.times[i] < 300)


c, a = seq(("hit", 5), ("hit", 5), ("hit", 305), ("getHits", 305))
print("sai:", run_commands(SaiRing, c, a), "| đúng:", run_commands(HitCounter, c, a))

sai: [None, None, None, None, 3] | đúng: [None, None, None, None, 1]


**Vì sao sai:** ô `5` chứa 2 hit của giây `5`; khi hit giây `305` (cùng ô `305 % 300 = 5`) ta phải **xóa dữ liệu cũ** (số lượt = `1`), nhưng code sai chỉ `+= 1` nên ô có `3` lượt và `getHits(305)` trả `3` thay vì `1`. **Cách sửa:** `if times[i] != t: times[i] = t; hits[i] = 1 else: hits[i] += 1`.

### Lỗi 4: Dùng `list.pop(0)` hoặc `while` không kiểm tra rỗng

In [29]:
class SaiRong:
    def __init__(self):
        self.q = deque()

    def hit(self, timestamp):
        self.q.append(timestamp)

    def getHits(self, timestamp):
        while self.q[0] <= timestamp - 300:       # q có thể rỗng -> IndexError
            self.q.popleft()
        return len(self.q)


c, a = seq(("hit", 1), ("getHits", 1000))
try:
    run_commands(SaiRong, c, a)
except IndexError as e:
    print("IndexError:", e)
print("đúng:", run_commands(HitCounter, c, a))

IndexError: deque index out of range
đúng: [None, None, 0]


**Vì sao sai:** hit duy nhất bị bỏ, hàng đợi rỗng, nhưng vòng `while` vẫn đọc `q[0]` ⇒ `IndexError`. **Cách sửa:** luôn viết `while self.q and self.q[0] <= ...` (kiểm tra **rỗng trước**, vì `and` dừng ngay khi vế trái sai). Còn `list.pop(0)` thì đúng nhưng chậm (`O(n)` mỗi lần), nên với dữ liệu lớn hãy dùng `deque`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "**Đếm trong cửa sổ thời gian gần nhất**" + thời gian **không giảm** ⇒ **hàng đợi trượt**: `append` vào cuối, `popleft` các phần tử hết hạn ở đầu.
- Danh sách đã sắp xếp + "đếm phần tử `>= x`" ⇒ `len(a) - bisect_left(a, x)`.
- Nhiều sự kiện cùng thời điểm ⇒ **nén** thành `(thời_điểm, số_lượng)` để bộ nhớ không phụ thuộc số sự kiện.
- Cửa sổ cố định `W` đơn vị thời gian ⇒ **mảng vòng** `W` ô, chỉ số `t % W`, nhớ lưu kèm **nhãn thời gian** của ô để biết ô còn hạn không.
- Luôn viết rõ cửa sổ là khoảng **nửa mở** `(t − W, t]` và kiểm tra bằng biên `t − W` / `t − W + 1`.
- Bài thiết kế: đặt **trạng thái** vào `self`, viết `run_commands` để chạy chuỗi lệnh ví dụ, và dùng oracle brute-force để so sánh ngẫu nhiên.

## 15. Biến thể của bài

1. **Cửa sổ `W` tùy chọn** (không cố định 300) — tổng quát hóa lớp.
2. **Number of Recent Calls (933)** — đếm ping trong `[t − 3000, t]` (cửa sổ **đóng** hai đầu, chỉ khác độ dài và biên).
3. **Logger Rate Limiter (359)** — mỗi thông điệp chỉ in nếu lần in trước cách `>= 10` giây (dùng `dict` thông điệp → thời điểm).
4. **Moving Average from Data Stream (346)** — trung bình của `size` giá trị gần nhất (hàng đợi + tổng chạy).
5. **Tweet Counts Per Frequency (1348)** — đếm theo phút/giờ/ngày trong khoảng bất kỳ (danh sách sắp xếp + `bisect`).
6. **Hit counter có xóa/hủy** hoặc **timestamp không đơn điệu** (khi đó hàng đợi không còn đúng; cần `bisect.insort` hoặc cây cân bằng).

In [30]:
class WindowCounter:
    """Hit counter với cửa sổ W tùy chọn: đếm hit có ts trong (t - W, t]."""

    def __init__(self, window):
        self.window = window
        self.q = deque()
        self.total = 0

    def hit(self, t, count=1):
        if self.q and self.q[-1][0] == t:
            self.q[-1][1] += count
        else:
            self.q.append([t, count])
        self.total += count

    def get(self, t):
        while self.q and self.q[0][0] <= t - self.window:
            self.total -= self.q.popleft()[1]
        return self.total


wc = WindowCounter(10)
wc.hit(1); wc.hit(5, count=3)
print(wc.get(10))                                        # ts > 0: 1 + 3 = 4
wc.hit(11)
print(wc.get(11), wc.get(15), wc.get(16), wc.get(21))    # 4 1 1 0
assert [wc.get(21)] == [0]

# Number of Recent Calls (933): ping(t) đếm trong [t-3000, t] -> bỏ khi ts < t - 3000
class RecentCounter:
    def __init__(self):
        self.q = deque()

    def ping(self, t):
        self.q.append(t)
        while self.q[0] < t - 3000:
            self.q.popleft()
        return len(self.q)


rc = RecentCounter()
print([rc.ping(t) for t in [1, 100, 3001, 3002]])        # [1, 2, 3, 3]

4
4 1 1 0
[1, 2, 3, 3]


Giải thích `wc` (cửa sổ `W = 10`, hit tại `1`, `5` (3 lượt), rồi `11`): `get(10)` ⇒ `ts > 0`: `1 + 3 = 4`; sau `hit(11)`, `get(11)` ⇒ `ts > 1`: bỏ giây `1`, còn `5` (3 lượt) và `11` (1 lượt) ⇒ `4`; `get(15)` ⇒ `ts > 5`: bỏ giây `5`, còn `11` ⇒ `1`; `get(16)` ⇒ `ts > 6` ⇒ `1`; `get(21)` ⇒ `ts > 11` ⇒ `0`.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time (`hit` / `getHits`) | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Lưu hết, đếm lại | duyệt mọi hit | `O(1)` / `O(n)` | `O(n)` | hiểu đề, làm oracle |
| 2. `deque` bỏ quá hạn | popleft phần tiền tố hết hạn | `O(1)` / khấu hao `O(1)` | `O(h)` | **nên nộp** khi hit ít |
| 3. `bisect` | tìm ranh giới trên list tăng dần | `O(1)` / `O(log n)` | `O(n)` | code ngắn, cần hỏi mốc cũ |
| 4. Mảng vòng 300 ô | bucket theo `t % 300` | `O(1)` / `O(300)` | `O(300)` | follow-up, bộ nhớ cố định |
| 5. `deque` nén | `(giây, số lượt)` + tổng chạy | `O(1)` / khấu hao `O(1)` | `O(min(h, 300))` | follow-up, hit rất nhiều mỗi giây |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: (a) `d = deque([1, 2, 3]); d.append(4); d.popleft(); print(list(d))`; (b) `bisect_left([1, 2, 3, 300], 2)` và `bisect_left([1, 2, 3, 300], 4)`; (c) `305 % 300`; (d) hit tại `10, 20, 310` rồi `getHits(310)`.

**Bài 2 (tính tay).** Các lệnh theo thứ tự: `hit(1)`, `hit(1)`, `hit(2)`, `hit(100)`, `hit(301)`, `getHits(301)`, `hit(302)`, `getHits(302)`. Tính hai kết quả `getHits` (nhớ cửa sổ `(t − 300, t]`), liệt kê hit nào còn sống.

**Bài 3 (điền TODO).** Hoàn thành lớp `HitCounterTodo` theo Cách 2 (`deque`).

In [31]:
class HitCounterTodo:
    def __init__(self):
        # TODO 1: tạo self.q là một deque rỗng
        pass

    def hit(self, timestamp):
        # TODO 2: thêm timestamp vào cuối hàng đợi
        pass

    def getHits(self, timestamp):
        # TODO 3: while hàng đợi không rỗng và phần tử đầu <= timestamp - 300: popleft
        # TODO 4: trả về số phần tử còn lại
        pass

# KHÔNG tạo đối tượng / gọi ở đây để Run All không lỗi (TODO chưa điền).

**Bài 4 (tự viết lại).** Viết `HitCounterRing2` theo Cách 4 (mảng vòng) **nhưng với cửa sổ `W` truyền vào hàm dựng** (`HitCounterRing2(window)`), dùng `t % window`.

**Bài 5 (biến thể / mini challenge).** (a) Viết `RecentCounter933` (`ping(t)` đếm cuộc gọi trong khoảng `[t − 3000, t]`, cửa sổ **đóng**), kiểm tra bằng ví dụ `[1, 100, 3001, 3002] → [1, 2, 3, 3]`. (b) Viết `HitCounterAnyOrder` chấp nhận `hit(t)` với `t` **không** theo thứ tự (gợi ý: dùng `bisect.insort`) và `getHits(t)` đếm đúng bằng `bisect`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: các đáp án là `[2, 3, 4]`; `1` và `3`; `5`; `2`. Bài 3: nhớ `from collections import deque` và `self.q = deque()`.
- **Mức 2 (vừa):** Bài 2: tại `t = 301` ngưỡng là `ts > 1` (hai hit giây `1` đều bị loại); tại `t = 302` ngưỡng là `ts > 2` (giây `2` cũng bị loại). Bài 4: ô `i = t % window`; hai thuộc tính `times` và `hits`; điều kiện còn hạn `t - times[i] < window`.
- **Mức 3 (sâu):** Bài 5(a): `while self.q[0] < t - 3000: popleft` (cửa sổ đóng nên bỏ khi **nhỏ hơn** `t − 3000`). 5(b): `insort(self.ts, t)` giữ danh sách sắp xếp; `getHits(t) = bisect_right(ts, t) - bisect_left(ts, t - 299)`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [32]:
from bisect import bisect_right, insort

# Bài 1
d = deque([1, 2, 3]); d.append(4); d.popleft()
assert list(d) == [2, 3, 4]
assert bisect_left([1, 2, 3, 300], 2) == 1 and bisect_left([1, 2, 3, 300], 4) == 3
assert 305 % 300 == 5
c, a = seq(("hit", 10), ("hit", 20), ("hit", 310), ("getHits", 310))
assert run_commands(HitCounter, c, a)[-1] == 2          # ts > 10: chỉ 20 và 310

# Bài 2 (đúng thứ tự thời gian): hit 1,1,2,100,301 -> getHits(301) -> hit 302 -> getHits(302)
c, a = seq(("hit", 1), ("hit", 1), ("hit", 2), ("hit", 100), ("hit", 301), ("getHits", 301), ("hit", 302), ("getHits", 302))
out = run_commands(HitCounter, c, a)
assert out[-3] == 3 and out[-1] == 3, out        # t=301: ts>1 -> {2,100,301}=3 ; t=302: ts>2 -> {100,301,302}=3
for impl in impls.values():
    assert run_commands(impl, c, a) == out


# Bài 3
class HitCounterTodoAns:
    def __init__(self):
        self.q = deque()

    def hit(self, timestamp):
        self.q.append(timestamp)

    def getHits(self, timestamp):
        while self.q and self.q[0] <= timestamp - 300:
            self.q.popleft()
        return len(self.q)


assert run_commands(HitCounterTodoAns, commands, arguments) == expected_example


# Bài 4
class HitCounterRing2:
    def __init__(self, window=300):
        self.window = window
        self.times = [0] * window
        self.hits = [0] * window

    def hit(self, timestamp):
        i = timestamp % self.window
        if self.times[i] != timestamp:
            self.times[i], self.hits[i] = timestamp, 1
        else:
            self.hits[i] += 1

    def getHits(self, timestamp):
        return sum(self.hits[i] for i in range(self.window) if timestamp - self.times[i] < self.window)


assert run_commands(HitCounterRing2, commands, [[]] + arguments[1:]) == expected_example
assert run_commands(HitCounterRing2, ["R", "hit", "hit", "getHits", "getHits"], [[10], [1], [5], [10], [11]]) == [None, None, None, 2, 1]


# Bài 5(a)
class RecentCounter933:
    def __init__(self):
        self.q = deque()

    def ping(self, t):
        self.q.append(t)
        while self.q[0] < t - 3000:
            self.q.popleft()
        return len(self.q)


assert run_commands(RecentCounter933, ["RecentCounter", "ping", "ping", "ping", "ping"], [[], [1], [100], [3001], [3002]]) == [None, 1, 2, 3, 3]


# Bài 5(b)
class HitCounterAnyOrder:
    def __init__(self):
        self.ts = []

    def hit(self, timestamp):
        insort(self.ts, timestamp)

    def getHits(self, timestamp):
        return bisect_right(self.ts, timestamp) - bisect_left(self.ts, timestamp - 299)


c, a = seq(("hit", 50), ("hit", 10), ("hit", 400), ("getHits", 310), ("getHits", 400), ("hit", 20), ("getHits", 310))
assert run_commands(HitCounterAnyOrder, c, a) == [None, None, None, None, 1, 1, None, 2]
print("Tất cả đáp án đều PASS")

Tất cả đáp án đều PASS


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (cửa sổ trượt bằng hàng đợi)

```python
from collections import deque


class HitCounter:

    def __init__(self):
        self.q = deque()

    def hit(self, timestamp: int) -> None:
        self.q.append(timestamp)

    def getHits(self, timestamp: int) -> int:
        while self.q and self.q[0] <= timestamp - 300:
            self.q.popleft()
        return len(self.q)
```

**Checklist**

- [ ] Trạng thái nằm trong `self` (khởi tạo ở `__init__`), không phải biến cục bộ?
- [ ] Cửa sổ là `(t − 300, t]`: bỏ khi `ts <= t − 300`, giữ khi `ts >= t − 299`? Đã thử biên `299/300/301`?
- [ ] `getHits` tự dọn hit quá hạn (không chỉ `hit`)? Có `self.q and ...` trước `self.q[0]`?
- [ ] Nhiều hit cùng `timestamp` đều được đếm (không dùng `set`)?
- [ ] Với follow-up: dùng mảng vòng `t % 300` (nhớ ghi đè ô cũ) hoặc nén `(giây, số lượt)`?
- [ ] Đã chạy ví dụ đề `[null,null,null,null,3,null,4,3]` bằng bộ chạy chuỗi lệnh và so với oracle ngẫu nhiên?